In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.impute import SimpleImputer

In [16]:
ames = fetch_openml(name="house_prices", as_frame=True, parser="auto")
X = ames.data
y = ames.target
df = pd.concat([X,y],axis = 1)
df.to_csv("../data/ames_housing.csv", index = False)

obj_cols = X.select_dtypes(include=['object']).columns
X[obj_cols] = X[obj_cols].astype('category')



/tmp/ipykernel_203276/309049753.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[obj_cols] = X[obj_cols].astype('category')


In [26]:
df[df.select_dtypes(include=["number"]).columns].isnull().sum()

Id                 0
MSSubClass         0
LotFrontage      259
LotArea            0
OverallQual        0
OverallCond        0
YearBuilt          0
YearRemodAdd       0
MasVnrArea         8
BsmtFinSF1         0
BsmtFinSF2         0
BsmtUnfSF          0
TotalBsmtSF        0
1stFlrSF           0
2ndFlrSF           0
LowQualFinSF       0
GrLivArea          0
BsmtFullBath       0
BsmtHalfBath       0
FullBath           0
HalfBath           0
BedroomAbvGr       0
KitchenAbvGr       0
TotRmsAbvGrd       0
Fireplaces         0
GarageYrBlt       81
GarageCars         0
GarageArea         0
WoodDeckSF         0
OpenPorchSF        0
EnclosedPorch      0
3SsnPorch          0
ScreenPorch        0
PoolArea           0
MiscVal            0
MoSold             0
YrSold             0
SalePrice          0
dtype: int64

In [15]:
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer,make_column_selector
from sklearn.preprocessing import StandardScaler

In [16]:
from sklearn.model_selection import train_test_split

In [17]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

In [18]:
ct = ColumnTransformer(
    transformers=[
        # 데이터 타입이 숫자형(정수, 실수)인 열만 자동으로 골라서 스케일링 적용
        ('num_scaler', StandardScaler(), make_column_selector(dtype_include=np.number))
    ],
    remainder='passthrough'
)

ct.set_output(transform="pandas")


ColumnTransformer(remainder='passthrough',
                  transformers=[('num_scaler', StandardScaler(),
                                 <sklearn.compose._column_transformer.make_column_selector object at 0x7bd40ebba7b0>)])

In [ ]:
y.min(), y.max()

(34900, 755000)

In [19]:
X_train_scaled = ct.fit_transform(X_train)
X_val_scaled = ct.transform(X_val)
X_test_scaled = ct.transform(X_test)  


In [20]:
y_train_scaled = np.log1p(y_train)
y_val_scaled = np.log1p(y_val)
y_test_scaled = np.log1p(y_test)  


In [21]:
import lightgbm as lgb

In [22]:
from sklearn.metrics import mean_squared_error, r2_score

In [23]:
model = lgb.LGBMRegressor(learning_rate=0.05, n_estimators=2000, random_state=42)
model.fit(X_train_scaled, y_train_scaled)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000347 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3247
[LightGBM] [Info] Number of data points in the train set: 934, number of used features: 75
[LightGBM] [Info] Start training from score 12.030005


LGBMRegressor(learning_rate=0.05, n_estimators=2000, random_state=42)

In [24]:
preds = model.predict(X_test_scaled)
rmse = np.sqrt(mean_squared_error(y_test_scaled, preds))
r2 = r2_score(y_test_scaled,preds)
print(f"RMSE: {rmse:.4f} \n"
        f"R2 : {r2:.4f} "
)

RMSE: 0.1359 
R2 : 0.9010 
